# Solving Trusses with Z3


A **truss** is a rigid engineering structure made up of long, slender members connected at their ends. Trusses are commonly used to span large distances with a strong, lightweight structure. Typical applications include bridges, roof structures, and pylons.

In a "simple truss", all joints are modeled as frictionless pins, and loads are only applied at the joints. Because of this, the members in a truss are strictly **two-force members**—they are only subjected to axial forces (either tension or compression), with no bending moments.

To "solve" a truss, we use the principles of static equilibrium. Since the entire structure is not moving, every individual joint must also be in equilibrium. This means the sum of forces in the x-direction ($\sum F_x$) and the y-direction ($\sum F_y$) at every joint must equal exactly zero. We will model this rule as a constraint satisfaction problem.

In this notebook, we will
look at SMT solvers and see how they can be used to formalize and solve trusses described above.


**Instructions:**
1. To get started, click on File on the top left and click "Save a copy in Drive."
This will give you an editable version of this document that you can use.
2. If you press `CMD`+`Enter` it runs the cell, and if you press `Shift`+`Enter` it runs the cell and goes to the next one.
3. Make sure you run all cells as you go through the notebook; some cells will not work properly unless the previous one
has been run too.
4. If you disconnect or are inactive for some time you should run all of the cells again.

## 0. Preliminaries (you should run this cell but there is no need to read it)

In [ ]:
!pip install z3-solver
!pip install git+https://github.com/crrivero/FormalMethodsTasting.git#subdirectory=core
from z3 import *
from tofmcore import visualize_truss_solution
import networkx as nx
import itertools
from IPython.display import clear_output
clear_output()

## Encoding constraints in Z3

The goal of this notebook is to teach you about formal methods;
particularly, how you can use existing formal verification tools
(in this case, Z3) to analyze and solve your own problems.
Before we get started, let's look at some basic things we can do with Z3.

### Reals

Let's use Z3 to solve problems involving reals. Let's start with something simple: find $x$ such that

$$2x + 5 = 10$$

In [ ]:
# Initialize variables

x = Real('x') # declairing that x is an integer named 'x'

# Initialize Z3 solver
s = Solver()

s.add( 2*x + 5 == 10 ) # add the equation

print(s)
print(s.check())
print(s.model())

Now let's try to check whether that's the only solution. We can do this by adding the following constraint to the solver:

$$x \not= \frac{5}{2}$$

If the solver returns "**unsat**" then $x=\frac{5}{2}$ is the only solution.
Try it yourself by completing the code in the cell below.

In [ ]:
s.add( x == 5/2 ) # REPLACE THIS LINE
s.check()

**Note that if we were to run s.model() now we would get an error.**

## Modeling and Solving the Truss Problem
We are going to solve a simple 3-bar truss forming a right triangle.

**The Setup:**

* **Joint A** is at origin `(0, 0)`. It is secured to the wall with a **pin support**, meaning it provides reaction forces in both X and Y directions ($A_x$ and $A_y$).
* **Joint B** is at `(3, 0)`. It rests on a **roller support**, meaning it can slide in X, but provides a vertical reaction force ($B_y$).
* **Joint C** is at `(0, 4)`. It is a free joint experiencing an external load. Let's assume a force of $100$ N pushing right (positive X) and $50$ N pushing down (negative Y).

**The Members:**

* Member **AB** (Length = 3)
* Member **AC** (Length = 4)
* Member **BC** (Length = 5, forming a 3-4-5 right triangle)

Let's write the equilibrium equations ($\sum F_x = 0$ and $\sum F_y = 0$) for every joint, assuming that internal forces pull *away* from the joint (Tension is positive, compression is negative).


#### 1. Create a fresh solver


In [ ]:
truss_solver = Solver()

#### 2. Declare Variables


In [ ]:
# Internal forces in the members
T_AB = Real('T_AB')
T_AC = Real('T_AC')
T_BC = Real('T_BC')

# Reaction forces at the supports
A_x = Real('A_x')
A_y = Real('A_y')
B_y = Real('B_y')

#### 3. Add Constraints (Method of Joints)

In [ ]:
# --- JOINT A (0, 0) ---
# Forces acting on A: Reaction A, T_AB (horizontal), T_AC (vertical)
truss_solver.add(A_x + T_AB == 0)      # Sum of F_x = 0
truss_solver.add(A_y + T_AC == 0)      # Sum of F_y = 0

# --- JOINT B (3, 0) ---
# Forces acting on B: Reaction B_y, T_AB (pulling left), T_BC (pulling up-left towards C)
# The vector from B to C is (-3, 4). The unit vector components are X: -3/5, Y: 4/5.
truss_solver.add(-T_AB + T_BC * (-3.0/5.0) == 0)   # Sum of F_x = 0
truss_solver.add(B_y + T_BC * (4.0/5.0) == 0)      # Sum of F_y = 0

# --- JOINT C (0, 4) ---
# Forces acting on C: External load (100N right, 50N down), T_AC (pulling down), T_BC (pulling down-right towards B)
# The vector from C to B is (3, -4). The unit vector components are X: 3/5, Y: -4/5.
truss_solver.add(100 + T_BC * (3.0/5.0) == 0)      # Sum of F_x = 0
truss_solver.add(-50 - T_AC + T_BC * (-4.0/5.0) == 0)  # Sum of F_y = 0

#### 4. Solve the Truss


In [ ]:
# Check if solution exists
print( truss_solver.check() )

In [ ]:
# View solution
solution = truss_solver.model()
print( solution )

To better understand what is happening structurally, we can plot the truss. We will extract the solved forces and map them to our nodes.

* **Blue lines** will represent members in **tension** (pulling).
* **Red lines** will represent members in **compression** (pushing).
* Line thickness will represent the magnitude of the force.

In [ ]:
f_AB = float(solution[T_AB].as_fraction())
f_AC = float(solution[T_AC].as_fraction())
f_BC = float(solution[T_BC].as_fraction())
visualize_truss_solution( f_AB, f_AC, f_BC )


### Congratulations! You just used an SMT solver to solve trusses!


####If you'd like to continue your Z3 journey, you can start with this guide to learn more:
https://ericpony.github.io/z3py-tutorial/guide-examples.htm